In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [3]:
import pandas as pd

from src.preprocessing import preprocess_data
from src.features import add_structural_features

In [10]:
df = pd.read_csv("../data/train.csv")

df = preprocess_data(df)
df = add_structural_features(df)

In [11]:
label_map = {
    "A": 0,
    "B": 1,
    "tie": 2
}

df["target"] = df["label"].map(label_map)

In [12]:
#train validation test 8:1:1
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=714,
    stratify=df["target"] #stratified split
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=714,
    stratify=temp_df["target"]
)

In [13]:
print("train:", train_df.shape)
print("validation:", val_df.shape)
print("test:", test_df.shape)

print("Train label distribution:")
print(train_df["label"].value_counts(normalize=True))

print("\nValidation label distribution:")
print(val_df["label"].value_counts(normalize=True))

print("\nValidation label distribution:")
print(val_df["label"].value_counts(normalize=True))

print("\nTest label distribution:")
print(test_df["label"].value_counts(normalize=True))

train: (45981, 25)
validation: (5748, 25)
test: (5748, 25)
Train label distribution:
A      0.349079
B      0.341902
tie    0.309019
Name: label, dtype: float64

Validation label distribution:
A      0.348991
B      0.342032
tie    0.308977
Name: label, dtype: float64

Validation label distribution:
A      0.348991
B      0.342032
tie    0.308977
Name: label, dtype: float64

Test label distribution:
A      0.349165
B      0.341858
tie    0.308977
Name: label, dtype: float64


# TF-IDF

In [14]:
train_all_text = pd.concat([
    train_df["prompt_text"],
    train_df["a_text"],
    train_df["b_text"]
], ignore_index=True)

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=3, #minimum document frequency (absolute count)
    max_df=0.95 #maximum document frequency (proportion/ratio)
)

In [16]:
tfidf.fit(train_all_text)

TfidfVectorizer(max_df=0.95, max_features=10000, min_df=3, ngram_range=(1, 2))

In [17]:
print(len(tfidf.vocabulary_))
print(tfidf.get_feature_names_out()[:20])

10000
['00' '00 00' '00 pm' '000' '000 000' '01' '02' '03' '04' '05' '06' '07'
 '0d' '0f' '10' '10 000' '10 10' '10 15' '10 minutes' '10 the']


In [18]:
X_train_prompt = tfidf.transform(train_df["prompt_text"])
X_train_a = tfidf.transform(train_df["a_text"])
X_train_b = tfidf.transform(train_df["b_text"])

In [19]:
X_val_prompt = tfidf.transform(val_df["prompt_text"])
X_val_a = tfidf.transform(val_df["a_text"])
X_val_b = tfidf.transform(val_df["b_text"])

In [20]:
print(X_train_prompt.shape)
print(X_train_a.shape)
print(X_train_b.shape)

print(X_val_prompt.shape)
print(X_val_a.shape)
print(X_val_b.shape)

(45981, 10000)
(45981, 10000)
(45981, 10000)
(5748, 10000)
(5748, 10000)
(5748, 10000)


In [21]:
from scipy.sparse import hstack

X_train_text = hstack([
    X_train_prompt,
    X_train_a,
    X_train_b
])

X_val_text = hstack([
    X_val_prompt,
    X_val_a,
    X_val_b
])

In [22]:
print(X_train_text.shape)
print(X_val_text.shape)

(45981, 30000)
(5748, 30000)


In [23]:
structural_features = [
    "a_chars",
    "b_chars",
    "length_diff",
    "abs_length_diff",
    "log_length_ratio",
    "normalized_length_diff",
    "prompt_turns",
    "is_multi_turn"
]

In [24]:
from scipy.sparse import csr_matrix

X_train_struct = csr_matrix(
    train_df[structural_features].values
)

X_val_struct = csr_matrix(
    val_df[structural_features].values
)

In [25]:
# combine shared-vocabulary TF-IDF representations with 8 structural features
X_train = hstack([
    X_train_text,
    X_train_struct
]).tocsr()

X_val = hstack([
    X_val_text,
    X_val_struct
]).tocsr()

In [26]:
print(X_train.shape)
print(X_val.shape)

(45981, 30008)
(5748, 30008)


In [27]:
y_train = train_df["target"].values
y_val = val_df["target"].values

print(y_train.shape)
print(y_val.shape)

(45981,)
(5748,)


# 0 baseline

In [28]:
import numpy as np
from sklearn.metrics import log_loss

uniform_pred = np.full(
    (len(y_val), 3),
    1 / 3
)

uniform_logloss = log_loss(
    y_val,
    uniform_pred,
    labels=[0, 1, 2]
)

print(f"Uniform baseline Log Loss: {uniform_logloss:.4f}")

Uniform baseline Log Loss: 1.0986


# Xgboost

In [29]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    n_estimators=120,
    learning_rate=0.08,
    max_depth=4,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.3,
    tree_method="hist",
    random_state=714,
    n_jobs=-1
)

In [30]:
xgb_model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=10
)

[0]	validation_0-mlogloss:1.09323
[10]	validation_0-mlogloss:1.06848
[20]	validation_0-mlogloss:1.05384
[30]	validation_0-mlogloss:1.04759
[40]	validation_0-mlogloss:1.04339
[50]	validation_0-mlogloss:1.04036
[60]	validation_0-mlogloss:1.03908
[70]	validation_0-mlogloss:1.03787
[80]	validation_0-mlogloss:1.03711
[90]	validation_0-mlogloss:1.03607
[100]	validation_0-mlogloss:1.03555
[110]	validation_0-mlogloss:1.03495
[119]	validation_0-mlogloss:1.03450


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.3, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.08, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=5, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=120, n_jobs=-1, num_class=3, ...)

In [31]:
val_pred = xgb_model.predict_proba(X_val)

xgb_logloss = log_loss(
    y_val,
    val_pred,
    labels=[0, 1, 2]
)

print(f"Uniform baseline: {uniform_logloss:.4f}")
print(f"XGBoost Log Loss: {xgb_logloss:.4f}")
print(f"Improvement: {uniform_logloss - xgb_logloss:.4f}")

Uniform baseline: 1.0986
XGBoost Log Loss: 1.0345
Improvement: 0.0641


In [33]:
print(val_pred[:5])
print(val_pred[:5].sum(axis=1))

[[0.41648796 0.27673173 0.30678028]
 [0.17169476 0.6315146  0.19679068]
 [0.54110354 0.23235655 0.22653992]
 [0.25507376 0.5063442  0.23858203]
 [0.27659068 0.36450958 0.35889974]]
[0.99999994 1.         1.         1.         1.        ]


In [37]:
#Hyperparameter Tuning 1

xgb_model_b = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    n_estimators=180, #original: 120
    learning_rate=0.05, #original: 0.08
    max_depth=3, #original: 4
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.3,
    tree_method="hist",
    random_state=714,
    n_jobs=-1
)

xgb_model_b.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=10
)

val_pred_b = xgb_model_b.predict_proba(X_val)

xgb_logloss_b = log_loss(
    y_val,
    val_pred_b,
    labels=[0, 1, 2]
)

print(f"Original Model: {xgb_logloss:.4f}")
print(f"Model B:        {xgb_logloss_b:.4f}")
print(f"Change:         {xgb_logloss - xgb_logloss_b:.4f}")

[0]	validation_0-mlogloss:1.09549
[10]	validation_0-mlogloss:1.07865
[20]	validation_0-mlogloss:1.06582
[30]	validation_0-mlogloss:1.05884
[40]	validation_0-mlogloss:1.05378
[50]	validation_0-mlogloss:1.05018
[60]	validation_0-mlogloss:1.04757
[70]	validation_0-mlogloss:1.04553
[80]	validation_0-mlogloss:1.04391
[90]	validation_0-mlogloss:1.04261
[100]	validation_0-mlogloss:1.04166
[110]	validation_0-mlogloss:1.04074
[120]	validation_0-mlogloss:1.03988
[130]	validation_0-mlogloss:1.03916
[140]	validation_0-mlogloss:1.03861
[150]	validation_0-mlogloss:1.03812
[160]	validation_0-mlogloss:1.03770
[170]	validation_0-mlogloss:1.03725
[179]	validation_0-mlogloss:1.03698
Original Model: 1.0345
Model B:        1.0370
Change:         -0.0025


In [38]:
#Hyperparameter Tuning 2

xgb_model_c = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    n_estimators=180, #original: 120
    learning_rate=0.05, #original: 0.08
    max_depth=4, 
    min_child_weight=3, #original: 5
    subsample=0.8,
    colsample_bytree=0.3,
    tree_method="hist",
    random_state=714,
    n_jobs=-1
)

xgb_model_c.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=10
)

val_pred_c = xgb_model_c.predict_proba(X_val)

xgb_logloss_c = log_loss(
    y_val,
    val_pred_c,
    labels=[0, 1, 2]
)

print(f"Original Model: {xgb_logloss:.4f}")
print(f"Model C:        {xgb_logloss_b:.4f}")
print(f"Change:         {xgb_logloss - xgb_logloss_b:.4f}")

[0]	validation_0-mlogloss:1.09519
[10]	validation_0-mlogloss:1.07696
[20]	validation_0-mlogloss:1.06330
[30]	validation_0-mlogloss:1.05594
[40]	validation_0-mlogloss:1.05099
[50]	validation_0-mlogloss:1.04751
[60]	validation_0-mlogloss:1.04476
[70]	validation_0-mlogloss:1.04289
[80]	validation_0-mlogloss:1.04130
[90]	validation_0-mlogloss:1.04006
[100]	validation_0-mlogloss:1.03924
[110]	validation_0-mlogloss:1.03841
[120]	validation_0-mlogloss:1.03780
[130]	validation_0-mlogloss:1.03694
[140]	validation_0-mlogloss:1.03653
[150]	validation_0-mlogloss:1.03601
[160]	validation_0-mlogloss:1.03567
[170]	validation_0-mlogloss:1.03533
[179]	validation_0-mlogloss:1.03513
Original Model: 1.0345
Model C:        1.0370
Change:         -0.0025


both hyperparameter tuning show the original model is better

## Ablation Study

In [39]:
from sklearn.base import clone

xgb_text_only = clone(xgb_model)

xgb_text_only.fit(
    X_train_text,
    y_train,
    eval_set=[(X_val_text, y_val)],
    verbose=10
)

val_pred_text = xgb_text_only.predict_proba(X_val_text)

text_logloss = log_loss(
    y_val,
    val_pred_text,
    labels=[0, 1, 2]
)

print(f"TF-IDF only:         {text_logloss:.4f}")
print(f"TF-IDF + structure:  {xgb_logloss:.4f}")
print(f"Feature improvement: {text_logloss - xgb_logloss:.4f}")

[0]	validation_0-mlogloss:1.09676
[10]	validation_0-mlogloss:1.08137
[20]	validation_0-mlogloss:1.07222
[30]	validation_0-mlogloss:1.06669
[40]	validation_0-mlogloss:1.06260
[50]	validation_0-mlogloss:1.05938
[60]	validation_0-mlogloss:1.05699
[70]	validation_0-mlogloss:1.05468
[80]	validation_0-mlogloss:1.05284
[90]	validation_0-mlogloss:1.05158
[100]	validation_0-mlogloss:1.05050
[110]	validation_0-mlogloss:1.04970
[119]	validation_0-mlogloss:1.04885
TF-IDF only:         1.0488
TF-IDF + structure:  1.0345
Feature improvement: 0.0144


structural features further reduce validation Log Loss by 0.0144, showing that response length and turn structure provide useful information beyond TF-IDF text features

## model performance on the test data

In [41]:
X_test_prompt = tfidf.transform(test_df["prompt_text"])
X_test_a = tfidf.transform(test_df["a_text"])
X_test_b = tfidf.transform(test_df["b_text"])

X_test_text = hstack([
    X_test_prompt,
    X_test_a,
    X_test_b
])

X_test_struct = csr_matrix(
    test_df[structural_features].values
)

X_test = hstack([
    X_test_text,
    X_test_struct
]).tocsr()

y_test = test_df["target"].values

In [42]:
test_pred = xgb_model.predict_proba(X_test)

test_logloss = log_loss(
    y_test,
    test_pred,
    labels=[0, 1, 2]
)

print(f"Validation Log Loss: {xgb_logloss:.4f}")
print(f"Test Log Loss:       {test_logloss:.4f}")

Validation Log Loss: 1.0345
Test Log Loss:       1.0327


# Model Slice Analysis by Response Length Gap

In [34]:
val_analysis = val_df.copy()

val_analysis["pred"] = val_pred.argmax(axis=1)
val_analysis["correct"] = (
    val_analysis["pred"] == val_analysis["target"]
)

val_analysis["abs_norm_length_diff"] = (
    val_analysis["normalized_length_diff"].abs()
)

val_analysis["length_gap_group"] = pd.cut(
    val_analysis["abs_norm_length_diff"],
    bins=[0, 0.05, 0.15, 0.30, 1.0],
    labels=[
        "very similar",
        "similar",
        "moderate",
        "large"
    ],
    include_lowest=True
)

In [35]:
val_analysis.groupby(
    "length_gap_group",
    observed=True
)["correct"].mean()

length_gap_group
very similar    0.413437
similar         0.419920
moderate        0.456103
large           0.524159
Name: correct, dtype: float64

In [36]:
from sklearn.metrics import log_loss

for group in val_analysis["length_gap_group"].dropna().unique():
    mask = val_analysis["length_gap_group"] == group

    group_loss = log_loss(
        y_val[mask],
        val_pred[mask],
        labels=[0, 1, 2]
    )

    print(
        group,
        "n =", mask.sum(),
        "log loss =", round(group_loss, 4)
    )

large n = 2318 log loss = 0.9909
moderate n = 1401 log loss = 1.0544
very similar n = 774 log loss = 1.062
similar n = 1255 log loss = 1.0758


### Slice Analysis Summary

The XGBoost baseline performs substantially better when the length difference between Response A and Response B is large, while performance deteriorates as the two responses become more similar in length. In particular, the `similar` and `very similar` groups show noticeably higher Log Loss than the `large`-gap group.

This suggests that the TF-IDF + XGBoost baseline relies considerably on surface-level cues such as response length and lexical patterns. When these structural differences become less informative, the model has greater difficulty distinguishing human preference.

Therefore, similar-length response pairs represent an important source of model error and motivate the next modeling stage: sequence-based LSTM and Transformer models, which can capture word order, contextual dependencies, and richer semantic information beyond TF-IDF and structural features.